In [1]:
import json
import pandas as pd


# Load results
with open("../results/master_results.json") as f:
    data = json.load(f)
     
# Convert to DataFrame (each experiment = row, metrics = columns)
df = pd.DataFrame(data).T
df.index.name = "experiment"
df = df.reset_index()

# Display as table
df

,experiment,top_k_fraction,avg_delta,variance_delta,sem_delta,accuracy,num_correct,total,mean_rank,mean_ranking_pct,sem_mean_ranking_pct,n_with_rank,within_top5_pct_count,fraction_within_top5_pct
0,Llama-3.2-3B__Temperature_lambada_top0.05,0.05,-4.954160,20.151400,0.259174,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-3B__Temperature_lambada_top0.1,0.10,-6.823024,18.421273,0.247799,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-3B__Temperature_lambada_top0.2,0.20,-8.917719,18.778210,0.250188,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.2-3B__Semantic_lambada_top0.05,0.05,-5.050690,18.781744,0.250212,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Llama-3.2-3B__Semantic_lambada_top0.1,0.10,-6.770932,18.357299,0.247368,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,gemma-3-4b-pt__IG_IWSLT2017DE_EN_top0.1,0.10,-7.774122,19.597956,0.139993,0.767000,767.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
312,gemma-3-4b-pt__IG_IWSLT2017DE_EN_top0.2,0.20,-9.639725,16.945172,0.130174,0.767000,767.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
313,gemma-3-4b-pt__Temperature_IWSLT2017DE_EN_top0.05,0.05,-4.476322,18.143260,0.134697,0.767000,767.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
314,gemma-3-4b-pt__Temperature_IWSLT2017DE_EN_top0.1,0.10,-6.223858,19.284095,0.138867,0.767000,767.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
def table_generator(model_name, dataset_names):
    def get_avg_sem(entry):
        """Extract avg_delta and sem_delta from entry."""
        if entry is None:
            return None, None
        return entry.get("avg_delta"), entry.get("sem_delta")

    def fmt_val_plus_sem(avg, sem):
        """Format avg ± sem."""
        if avg is None:
            return None
        if sem is not None:
            return f"{avg:.3f} ± {sem:.3f}"
        return f"{avg:.3f}"

    def trapz_aupc(vals, sems, fracs=[0.05, 0.1, 0.2]):
        """
        Compute trapezoidal AUPC and propagated SEM.
        
        AUPC = sum_i 0.5 * (f_{i+1} - f_i) * (v_i + v_{i+1})
        
        Uncertainty propagation:
        Var(AUPC) = sum_i [0.5*(f_{i+1}-f_i)]^2 * (sem_i^2 + sem_{i+1}^2)
        """
        if any(v is None for v in vals):
            return None, None
        
        aupc = 0.0
        var_aupc = 0.0
        for i in range(len(fracs) - 1):
            h = fracs[i+1] - fracs[i]
            w = 0.5 * h
            aupc += w * (vals[i] + vals[i+1])
            if all(s is not None for s in sems):
                var_aupc += w**2 * (sems[i]**2 + sems[i+1]**2)
        
        sem_aupc = var_aupc**0.5 if var_aupc > 0 else None
        return aupc, sem_aupc

    methods = {
        "random":            lambda k, f: "random_ablation" in k and f in k,
        "Integrated Grads":  lambda k, f: "IG" in k and f in k,
        "Input x Grad":      lambda k, f: "gradient_x_input" in k and f in k,
        "Semantic Scope":    lambda k, f: "Semantic" in k and f in k and "random_drop" not in k,
        "Temperature Scope": lambda k, f: "Temperature" in k and f in k and "random_drop" not in k,
        # "Fisher Scope k1024":  lambda k, f: "Fisher_k_256" in k and f in k and "random_drop" not in k,
        # "Fisher Scope k256":  lambda k, f: "Fisher_k_256" in k and f in k and "random_drop" not in k,
        # "Fisher Scope k64":  lambda k, f: "Fisher_k_64" in k and f in k and "random_drop" not in k,
        # "Fisher Scope k16":  lambda k, f: "Fisher_k_16" in k and f in k and "random_drop" not in k,
        # "Fisher Scope k4":   lambda k, f: "Fisher_k_4" in k and f in k and "random_drop" not in k,
        "Fisher Scope k1":   lambda k, f: "Fisher_k_1" in k and f in k and "Fisher_k_16" not in k,
    }

    fracs = [0.05, 0.1, 0.2]
    
    # Build one column per dataset
    dataset_columns = {}
    accuracies = {}
    for dataset_name in dataset_names:
        dataset_data = {k: v for k, v in data.items() 
                       if dataset_name in k and model_name in k}
        
        if dataset_data:
            accuracies[dataset_name] = next(iter(dataset_data.values()))['accuracy']

        col = {}
        for method_name, matcher in methods.items():
            vals, sems = [], []
            for frac in fracs:
                frac_str = f"top{frac}"
                key = next((k for k in dataset_data if matcher(k, frac_str)), None)
                avg, sem = get_avg_sem(dataset_data[key] if key else None)
                vals.append(avg)
                sems.append(sem)
            
            aupc, sem_aupc = trapz_aupc(vals, sems, fracs)
            col[method_name] = fmt_val_plus_sem(aupc, sem_aupc)
        
        dataset_columns[dataset_name] = col

    # Combine into single dataframe with one column per dataset
    result_table = pd.DataFrame(dataset_columns)
    result_table.index.name = "method"

    print(f"{model_name}")
    for dataset_name, acc in accuracies.items():
        print(f"  {dataset_name} — prediction accuracy: {acc}")
    
    return result_table

In [3]:
model_name = "Llama-3.2-1B"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)


Llama-3.2-1B
  lmbd1000 — prediction accuracy: 0.699
  IWSLT2017DE_EN — prediction accuracy: 0.726


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.261 ± 0.007,-0.298 ± 0.007
Integrated Grads,-1.095 ± 0.011,-0.935 ± 0.009
Input x Grad,-1.284 ± 0.011,-1.043 ± 0.010
Semantic Scope,-1.298 ± 0.011,-1.057 ± 0.010
Temperature Scope,-1.317 ± 0.011,-1.093 ± 0.009
Fisher Scope k1,-1.320 ± 0.010,-1.079 ± 0.010


In [4]:
model_name = "Llama-3.2-3B"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)


Llama-3.2-3B
  lmbd1000 — prediction accuracy: 0.775
  IWSLT2017DE_EN — prediction accuracy: 0.754


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.229 ± 0.007,-0.187 ± 0.006
Integrated Grads,-0.672 ± 0.010,-0.576 ± 0.009
Input x Grad,-1.119 ± 0.011,-0.768 ± 0.010
Semantic Scope,-1.162 ± 0.011,-0.777 ± 0.010
Temperature Scope,-1.173 ± 0.011,-0.760 ± 0.010
Fisher Scope k1,-1.175 ± 0.011,-0.797 ± 0.010


In [5]:
model_name = "Llama-3.1-8B"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)


Llama-3.1-8B
  lmbd1000 — prediction accuracy: 0.835
  IWSLT2017DE_EN — prediction accuracy: 0.715


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.294 ± 0.018,-0.218 ± 0.015
Integrated Grads,-1.013 ± 0.027,-0.791 ± 0.026
Input x Grad,-1.196 ± 0.026,-0.716 ± 0.024
Semantic Scope,-1.175 ± 0.026,-0.641 ± 0.024
Temperature Scope,-1.145 ± 0.027,-0.552 ± 0.023
Fisher Scope k1,-1.205 ± 0.026,-0.677 ± 0.024


In [6]:
model_name = "Qwen2.5-3B"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)

Qwen2.5-3B
  lmbd1000 — prediction accuracy: 0.725
  IWSLT2017DE_EN — prediction accuracy: 0.732


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.293 ± 0.008,-0.202 ± 0.009
Integrated Grads,-1.386 ± 0.011,-1.111 ± 0.015
Input x Grad,-1.392 ± 0.011,-1.113 ± 0.015
Semantic Scope,-1.336 ± 0.011,-1.010 ± 0.015
Temperature Scope,-1.554 ± 0.010,-1.201 ± 0.014
Fisher Scope k1,-1.410 ± 0.011,-1.148 ± 0.010


In [7]:
model_name = "Qwen2.5-1.5B"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)

Qwen2.5-1.5B
  lmbd1000 — prediction accuracy: 0.7
  IWSLT2017DE_EN — prediction accuracy: 0.736


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.349 ± 0.009,-0.250 ± 0.006
Integrated Grads,-1.714 ± 0.013,-1.354 ± 0.012
Input x Grad,-1.720 ± 0.013,-1.355 ± 0.012
Semantic Scope,-1.682 ± 0.013,-1.240 ± 0.012
Temperature Scope,-1.861 ± 0.012,-1.380 ± 0.012
Fisher Scope k1,-1.775 ± 0.012,-1.377 ± 0.012


In [8]:
model_name = "gemma-3-1b"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)

gemma-3-1b
  lmbd1000 — prediction accuracy: 0.592
  IWSLT2017DE_EN — prediction accuracy: 0.735


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.418 ± 0.009,-0.405 ± 0.008
Integrated Grads,-0.639 ± 0.011,-0.908 ± 0.011
Input x Grad,-1.561 ± 0.013,-1.356 ± 0.010
Semantic Scope,-1.635 ± 0.013,-1.302 ± 0.010
Temperature Scope,-1.662 ± 0.012,-1.290 ± 0.010
Fisher Scope k1,-1.670 ± 0.012,-1.393 ± 0.010


In [9]:
model_name = "gemma-3-4b"
dataset_names = ["lmbd1000", "IWSLT2017DE_EN"]

table_generator(model_name, dataset_names)

gemma-3-4b
  lmbd1000 — prediction accuracy: 0.752
  IWSLT2017DE_EN — prediction accuracy: 0.767


,lmbd1000,IWSLT2017DE_EN
method,,
random,-0.291 ± 0.008,-0.181 ± 0.006
Integrated Grads,-1.430 ± 0.015,-1.208 ± 0.011
Input x Grad,-1.704 ± 0.015,-1.171 ± 0.011
Semantic Scope,-1.775 ± 0.015,-1.055 ± 0.011
Temperature Scope,-1.778 ± 0.015,-0.993 ± 0.011
Fisher Scope k1,-1.814 ± 0.014,-1.196 ± 0.011
